In [6]:
# Install dependencies for Excel export (run this cell first if you get "No module named 'openpyxl'")
import subprocess
import sys
for pkg in ['pandas', 'openpyxl']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

In [7]:
"""
Classical Shor's Algorithm - Comprehensive Scaling Analysis
Extended data collection with extrapolation to RSA key sizes
Consistent 10 iterations per N with bit count tracking
"""

import math
import os
import numpy as np
import time
import random
import pandas as pd

# ============================================================================
# CORE FUNCTIONS
# ============================================================================

def gcd(a, b):
    """Compute GCD."""
    while b:
        a, b = b, a % b
    return a

def is_prime(n):
    """Check if n is prime."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def classical_order(x, N):
    """Find order r where x^r ≡ 1 (mod N)."""
    r = 1
    value = x % N
    operations = 0
    while value != 1:
        value = (value * x) % N
        r += 1
        operations += 1
        if r > N:  # Safety check
            return None, operations
    return r, operations

def factorize(N):
    """Get the prime factorization of N."""
    factors = []
    d = 2
    temp_N = N
    while d * d <= temp_N:
        while temp_N % d == 0:
            factors.append(d)
            temp_N //= d
        d += 1
    if temp_N > 1:
        factors.append(temp_N)
    return factors

def analyze_single_iteration(N, a):
    """Run one iteration of Shor's algorithm with base a."""
    result = {
        'success': False,
        'operations': 0,
        'failure_reason': None
    }
    
    # Check if coprime
    g = gcd(a, N)
    result['operations'] += 1
    
    if g != 1:
        result['success'] = True
        result['method'] = 'gcd'
        return result
    
    # Find order
    r, ops = classical_order(a, N)
    result['operations'] += ops
    
    if r is None:
        result['failure_reason'] = 'order_not_found'
        return result
    
    if r % 2 != 0:
        result['failure_reason'] = 'odd_period'
        return result
    
    # Check x^(r/2) ≡ -1 (mod N)
    x_r_half = pow(a, r // 2, N)
    result['operations'] += 1
    
    if x_r_half == N - 1:
        result['failure_reason'] = 'x^(r/2)=-1'
        return result
    
    # Extract factors
    p = gcd(x_r_half - 1, N)
    q = gcd(x_r_half + 1, N)
    result['operations'] += 2
    
    # Check if we found non-trivial factors
    if (p > 1 and p < N) or (q > 1 and q < N):
        result['success'] = True
        result['method'] = 'period_finding'
    else:
        result['failure_reason'] = 'trivial_factors'
    
    return result

def run_iterations(N, num_iterations=10):
    """Run iterations of Shor's algorithm with random bases."""
    # Get valid bases (limit to first 10000 to save time for large N)
    valid_bases = [a for a in range(2, min(N, 10000)) if gcd(a, N) == 1]
    
    if len(valid_bases) == 0:
        return None
    
    # Warm-up run
    random_base = random.choice(valid_bases)
    _ = analyze_single_iteration(N, random_base)
    
    # Run iterations with timing
    success_count = 0
    total_operations = 0
    
    start_time = time.perf_counter()
    
    for i in range(num_iterations):
        a = random.choice(valid_bases)
        result = analyze_single_iteration(N, a)
        total_operations += result['operations']
        if result['success']:
            success_count += 1
    
    total_time = (time.perf_counter() - start_time) * 1000  # ms
    
    return {
        'N': N,
        'num_bits': N.bit_length(),
        'num_digits': len(str(N)),
        'factors': factorize(N),
        'num_iterations': num_iterations,
        'num_valid_bases': len(valid_bases),
        'success_count': success_count,
        'success_rate': (success_count / num_iterations * 100),
        'total_operations': total_operations,
        'avg_operations': total_operations / num_iterations,
        'total_time_ms': total_time,
        'avg_time_per_iteration_ms': total_time / num_iterations
    }

def generate_test_numbers(min_n=15, max_n=10000000, num_samples=200):
    """
    Generate composite numbers across the range for testing.
    Uses logarithmic spacing for excellent coverage.
    """
    print(f"\nGenerating {num_samples} test numbers from {min_n:,} to {max_n:,}...")
    
    # Generate logarithmically spaced points
    log_min = np.log10(min_n)
    log_max = np.log10(max_n)
    log_points = np.linspace(log_min, log_max, num_samples)
    
    test_numbers = []
    
    for log_val in log_points:
        target = int(10 ** log_val)
        
        # Find nearest good composite number
        candidate = target if target % 2 == 1 else target + 1
        attempts = 0
        
        while attempts < 100:
            if candidate < 15:
                candidate = 15
                break
            
            # Skip if prime
            if is_prime(candidate):
                candidate += 2
                attempts += 1
                continue
            
            # Get factors
            factors = factorize(candidate)
            
            # Skip prime squares (problematic)
            if len(factors) == 2 and factors[0] == factors[1]:
                candidate += 2
                attempts += 1
                continue
            
            # Good composite found
            test_numbers.append(candidate)
            break
        
        if attempts >= 100:
            # Fallback: just use a known good composite nearby
            test_numbers.append(target + 6 if target % 2 == 1 else target + 7)
    
    # Remove duplicates and sort
    test_numbers = sorted(list(set(test_numbers)))
    
    print(f"Generated {len(test_numbers)} unique test numbers")
    print(f"Actual range: {min(test_numbers):,} to {max(test_numbers):,}")
    print(f"Bit range: {min(test_numbers).bit_length()} to {max(test_numbers).bit_length()} bits")
    
    return test_numbers

def extrapolate_to_rsa_sizes(results):
    """
    Extrapolate results to RSA key sizes (512, 1024, 2048, 4096 bits).
    """
    print("\n" + "="*100)
    print(" "*30 + "EXTRAPOLATION TO RSA KEY SIZES")
    print("="*100)
    
    # Fit power law: ops = a * N^b
    # Using log-log linear regression
    N_values = np.array([r['N'] for r in results])
    ops_values = np.array([r['avg_operations'] for r in results])
    
    # Log transform
    log_N = np.log10(N_values)
    log_ops = np.log10(ops_values)
    
    # Linear fit in log space
    coeffs = np.polyfit(log_N, log_ops, 1)
    exponent = coeffs[0]  # This is 'b' in N^b
    log_a = coeffs[1]     # This is log10(a)
    a = 10 ** log_a
    
    print(f"\nOBSERVED SCALING LAW:")
    print(f"  Operations ≈ {a:.2e} × N^{exponent:.3f}")
    print(f"  Complexity class: O(N^{exponent:.2f})")
    
    # Calculate R² (goodness of fit)
    predicted_log_ops = coeffs[0] * log_N + coeffs[1]
    residuals = log_ops - predicted_log_ops
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((log_ops - np.mean(log_ops))**2)
    r_squared = 1 - (ss_res / ss_tot)
    
    print(f"  R² (goodness of fit): {r_squared:.6f}")
    print(f"  {'Excellent' if r_squared > 0.99 else 'Good' if r_squared > 0.95 else 'Fair'} fit quality")
    
    # RSA key sizes
    rsa_sizes = {
        'RSA-512': 512,
        'RSA-1024': 1024,
        'RSA-2048': 2048,
        'RSA-4096': 4096
    }
    
    # Reference: use largest measured value
    ref_result = results[-1]
    ref_N = ref_result['N']
    ref_ops = ref_result['avg_operations']
    ref_time = ref_result['avg_time_per_iteration_ms'] / 1000  # Convert to seconds
    
    print(f"\nREFERENCE MEASUREMENT (largest N tested):")
    print(f"  N = {ref_N:,}")
    print(f"  Bits = {ref_result['num_bits']}")
    print(f"  Digits = {ref_result['num_digits']}")
    print(f"  Avg operations = {ref_ops:.2f}")
    print(f"  Avg time per iteration = {ref_time:.6f} seconds")
    
    print("\n" + "="*100)
    print("EXTRAPOLATION RESULTS:")
    print("="*100)
    print(f"\n{'Key Size':<15} {'Bits':<8} {'Digits':<10} {'Log₁₀(Ops)':<15} {'Time Estimate':<40}")
    print("-" * 100)
    
    # Time constants
    seconds_per_year = 365.25 * 24 * 3600
    age_of_universe_years = 13.8e9
    
    for name, bits in rsa_sizes.items():
        # Typical RSA number is close to 2^bits
        digits = int(bits * np.log10(2))  # Approximate decimal digits
        
        # Do ALL calculations in LOG SPACE to avoid overflow
        # log10(N_rsa) = bits * log10(2)
        log_N_rsa = bits * np.log10(2)
        
        # log10(predicted_ops) = log10(a) + exponent * log10(N_rsa)
        log_predicted_ops = log_a + exponent * log_N_rsa
        
        # For time estimate, use ratio in log space
        # log10(time) = log10(ops) + log10(time_ratio)
        log_time_ratio = np.log10(ref_time) - np.log10(ref_ops) if ref_ops > 0 else -9
        log_predicted_time_seconds = log_predicted_ops + log_time_ratio
        
        # Convert to years
        log_seconds_per_year = np.log10(seconds_per_year)
        log_predicted_years = log_predicted_time_seconds - log_seconds_per_year
        
        # Universe ages
        log_age_universe = np.log10(age_of_universe_years)
        log_universe_ages = log_predicted_years - log_age_universe
        
        # Format output
        ops_str = f"10^{log_predicted_ops:.1f}"
        
        if log_predicted_years < 0:
            time_str = f"~10^{log_predicted_time_seconds:.1f} seconds"
        elif log_predicted_years < 3:
            time_str = f"~10^{log_predicted_years:.1f} years"
        else:
            time_str = f"~10^{log_predicted_years:.1f} years (~10^{log_universe_ages:.1f} × universe)"
        
        print(f"{name:<15} {bits:<8} {digits:<10} {ops_str:<15} {time_str:<40}")
    
    print("-" * 100)
    
    print("\n" + "="*100)
    print("COMPARISON: CLASSICAL vs QUANTUM")
    print("="*100)
    
    # Compute RSA-2048 extrapolated values for accurate comparison text
    log_N_2048 = 2048 * np.log10(2)
    log_ops_2048 = log_a + exponent * log_N_2048
    log_time_2048 = log_ops_2048 + (np.log10(ref_time) - np.log10(ref_ops) if ref_ops > 0 else -9)
    log_years_2048 = log_time_2048 - np.log10(seconds_per_year)
    
    print(f"""
CLASSICAL SHOR'S ALGORITHM:
  • Observed complexity: O(N^{exponent:.2f})
  • Polynomial scaling makes RSA-2048 completely infeasible
  • For RSA-2048: ~10^{log_ops_2048:.0f} operations needed
  • Time required: ~10^{log_years_2048:.0f} years (incomprehensibly long)
  • Even with fastest supercomputers: Still impossible
  
QUANTUM SHOR'S ALGORITHM:
  • Theoretical complexity: O(log³N) = O(n³) where n = number of bits
  • For RSA-512:  ~512³  ≈ 134 million quantum gates
  • For RSA-1024: ~1024³ ≈ 1.07 billion quantum gates  
  • For RSA-2048: ~2048³ ≈ 8.59 billion quantum gates
  • For RSA-4096: ~4096³ ≈ 68.7 billion quantum gates
  
  With fault-tolerant quantum computer:
  • RSA-512:  Minutes to hours
  • RSA-1024: Hours to days
  • RSA-2048: Days to weeks
  • RSA-4096: Weeks to months
  
KEY INSIGHT:
  Classical: O(N^{exponent:.2f}) → Exponentially infeasible
  Quantum:   O(n³) where n=log(N) → Polynomial and feasible
  
SECURITY IMPLICATIONS:
  1. Current RSA is SAFE from classical computers
     (Would take longer than age of universe)
  
  2. RSA is VULNERABLE to quantum computers
     (Feasible with ~10,000 logical qubits + error correction)
  
  3. Post-quantum cryptography is URGENTLY needed
     (Quantum computers are actively being developed)
    """)
    
    return {
        'exponent': exponent,
        'coefficient': a,
        'log_coefficient': log_a,
        'r_squared': r_squared
    }

# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def main():
    """Run comprehensive analysis with extrapolation."""
    
    print("\n" + "="*100)
    print(" "*20 + "CLASSICAL SHOR'S ALGORITHM")
    print(" "*15 + "COMPREHENSIVE SCALING ANALYSIS WITH EXTRAPOLATION")
    print(" "*25 + "10 ITERATIONS PER N")
    print("="*100)
    
    # Configuration
    MIN_N = 15
    MAX_N = 10_000_000  # 10 million (7-8 digits)
    NUM_SAMPLES = 200   # More samples = smoother curve
    ITERATIONS_PER_N = 10  # Fixed: 10 iterations for every N
    
    print(f"\nConfiguration:")
    print(f"  Range: {MIN_N:,} to {MAX_N:,}")
    print(f"  Samples: {NUM_SAMPLES}")
    print(f"  Iterations per N: {ITERATIONS_PER_N} (consistent)")
    print(f"  Total iterations: {NUM_SAMPLES * ITERATIONS_PER_N:,}")
    
    # Generate test numbers
    N_values = generate_test_numbers(min_n=MIN_N, max_n=MAX_N, num_samples=NUM_SAMPLES)
    
    print(f"\nStarting analysis...")
    print(f"Estimated runtime: 5-15 minutes (depends on your hardware)")
    print()
    
    # Run analysis
    results = []
    start_total = time.time()
    
    for idx, N in enumerate(N_values, 1):
        # Progress indicator every 10 samples
        if idx % 10 == 0 or idx == 1:
            elapsed = time.time() - start_total
            percent = (idx / len(N_values)) * 100
            eta = (elapsed / idx) * (len(N_values) - idx) if idx > 0 else 0
            print(f"[{idx:3d}/{len(N_values)}] Progress: {percent:5.1f}% | "
                  f"Elapsed: {elapsed:6.1f}s | ETA: {eta:6.1f}s")
        
        result = run_iterations(N, num_iterations=ITERATIONS_PER_N)
        
        if result:
            results.append(result)
    
    total_elapsed = time.time() - start_total
    
    print(f"\n{'='*100}")
    print(f"Data collection complete!")
    print(f"Total time: {total_elapsed:.2f} seconds ({total_elapsed/60:.2f} minutes)")
    print(f"Successfully analyzed: {len(results)} numbers")
    print(f"{'='*100}")
    
    # Statistics
    avg_success_rate = np.mean([r['success_rate'] for r in results])
    total_operations = sum([r['total_operations'] for r in results])
    
    print(f"\nOVERALL STATISTICS:")
    print(f"  Numbers tested: {len(results)}")
    print(f"  Total iterations: {len(results) * ITERATIONS_PER_N:,}")
    print(f"  Total operations: {total_operations:,}")
    print(f"  Average success rate: {avg_success_rate:.1f}%")
    print(f"  Smallest N: {results[0]['N']:,} ({results[0]['num_bits']} bits)")
    print(f"  Largest N: {results[-1]['N']:,} ({results[-1]['num_bits']} bits)")
    
    # Extrapolation
    fit_params = extrapolate_to_rsa_sizes(results)
    
    # Print comprehensive results table
    print("\n" + "="*100)
    print(" "*35 + "FULL RESULTS TABLE")
    print("="*100)
    print(f"\n{'N':<12} {'Bits':<6} {'Digits':<8} {'Factors':<18} {'Success':<9} {'Rate%':<8} {'Avg Ops':<10} {'Time(ms)':<12}")
    print("-" * 100)
    
    for r in results:
        factors_str = 'x'.join(map(str, r['factors'][:3]))
        if len(r['factors']) > 3:
            factors_str += '...'
        
        print(f"{r['N']:<12,} {r['num_bits']:<6} {r['num_digits']:<8} {factors_str:<18} "
              f"{r['success_count']:<9}/10 {r['success_rate']:<8.1f} "
              f"{r['avg_operations']:<10.2f} {r['total_time_ms']:<12.3f}")
    
    print("-" * 100)
    
    # Export to Excel - save in "Comparison analysis circuits" folder
    cwd = os.getcwd()
    if os.path.basename(cwd) == "Comparison analysis circuits":
        excel_path = os.path.join(cwd, "Classical_Shor_Analysis_Results.xlsx")
    else:
        excel_path = os.path.join(cwd, "Comparison analysis circuits", "Classical_Shor_Analysis_Results.xlsx")
    try:
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            # Sheet 1: Full results
            df_results = pd.DataFrame([{
                'N': r['N'],
                'Bits': r['num_bits'],
                'Digits': r['num_digits'],
                'Factors': 'x'.join(map(str, r['factors'])),
                'Success_Count': r['success_count'],
                'Success_Rate_%': round(r['success_rate'], 1),
                'Avg_Operations': round(r['avg_operations'], 2),
                'Total_Time_ms': round(r['total_time_ms'], 3)
            } for r in results])
            df_results.to_excel(writer, sheet_name='Results', index=False)
            
            # Sheet 2: RSA extrapolation
            log_a = fit_params.get('log_coefficient', np.log10(fit_params['coefficient']))
            ref_ops = results[-1]['avg_operations']
            ref_time = results[-1]['avg_time_per_iteration_ms'] / 1000
            seconds_per_year = 365.25 * 24 * 3600
            rsa_data = []
            for name, bits in [('RSA-512', 512), ('RSA-1024', 1024), ('RSA-2048', 2048), ('RSA-4096', 4096)]:
                log_N_rsa = bits * np.log10(2)
                log_ops = log_a + fit_params['exponent'] * log_N_rsa
                log_time = log_ops + (np.log10(ref_time) - np.log10(ref_ops) if ref_ops > 0 else -9)
                log_years = log_time - np.log10(seconds_per_year)
                rsa_data.append({
                    'Key_Size': name,
                    'Bits': bits,
                    'Digits': int(bits * np.log10(2)),
                    'Log10_Operations': round(log_ops, 1),
                    'Log10_Time_Years': round(log_years, 1)
                })
            pd.DataFrame(rsa_data).to_excel(writer, sheet_name='RSA_Extrapolation', index=False)
            
            # Sheet 3: Summary
            df_summary = pd.DataFrame([
                {'Parameter': 'Scaling_Exponent', 'Value': round(fit_params['exponent'], 3)},
                {'Parameter': 'Coefficient', 'Value': fit_params['coefficient']},
                {'Parameter': 'R_squared', 'Value': round(fit_params['r_squared'], 6)},
                {'Parameter': 'Numbers_Tested', 'Value': len(results)},
                {'Parameter': 'N_Range', 'Value': f"{results[0]['N']:,} to {results[-1]['N']:,}"},
                {'Parameter': 'Bit_Range', 'Value': f"{results[0]['num_bits']} to {results[-1]['num_bits']}"},
            ])
            df_summary.to_excel(writer, sheet_name='Summary', index=False)
        
        print(f"\n📁 Data exported to: {os.path.abspath(excel_path)}")
    except ImportError as e:
        print(f"\n⚠️ Excel export skipped. Install: pip install pandas openpyxl")
        print(f"   Error: {e}")
    except Exception as e:
        print(f"\n⚠️ Excel export failed: {type(e).__name__}: {e}")
    
    # Data for plotting
    print("\n" + "="*100)
    print(" "*35 + "DATA FOR PLOTTING")
    print("="*100)
    
    print("\n📊 PLOT 1: Log-Log Plot (RECOMMENDED - Shows Power Law)")
    print("-" * 100)
    print("# X-axis: log₁₀(N), Y-axis: log₁₀(Operations)")
    print(f"# Expected: Straight line with slope ≈ {fit_params['exponent']:.2f}")
    print("log10_N =", [round(np.log10(r['N']), 4) for r in results])
    print("log10_Ops =", [round(np.log10(r['avg_operations']), 4) for r in results])
    
    print("\n📊 PLOT 2: Linear Scale (N vs Operations)")
    print("-" * 100)
    print("N =", [r['N'] for r in results])
    print("Avg_Ops =", [round(r['avg_operations'], 2) for r in results])
    
    print("\n📊 PLOT 3: Bits vs Operations (Good for RSA comparison)")
    print("-" * 100)
    print("# This directly compares to RSA key sizes")
    print("Bits =", [r['num_bits'] for r in results])
    print("Avg_Ops =", [round(r['avg_operations'], 2) for r in results])
    
    print("\n📊 PLOT 4: Log-Log Bits vs Operations")
    print("-" * 100)
    print("# Shows scaling in terms of bit length")
    print("Bits =", [r['num_bits'] for r in results])
    print("log10_Ops =", [round(np.log10(r['avg_operations']), 4) for r in results])
    
    print("\n📊 PLOT 5: Success Rate vs N")
    print("-" * 100)
    print("# Should show consistency (around 60%)")
    print("N =", [r['N'] for r in results])
    print("Success_Rate =", [round(r['success_rate'], 1) for r in results])
    
    print("\n📊 PLOT 6: Time vs N")
    print("-" * 100)
    print("N =", [r['N'] for r in results])
    print("Total_Time_ms =", [round(r['total_time_ms'], 3) for r in results])
    
    print("\n" + "="*100)
    print(" "*25 + "💡 KEY FINDINGS FOR YOUR REPORT")
    print("="*100)
    
    # Compute RSA-2048 extrapolated values for KEY FINDINGS
    log_ops_2048 = fit_params.get('log_coefficient', np.log10(fit_params['coefficient'])) + fit_params['exponent'] * (2048 * np.log10(2))
    log_time_2048 = log_ops_2048 + np.log10(results[-1]['avg_time_per_iteration_ms']/1000) - np.log10(results[-1]['avg_operations'])
    log_years_2048 = log_time_2048 - np.log10(365.25*24*3600)
    log_universe_ages_2048 = log_years_2048 - np.log10(13.8e9)
    
    print(f"""
1. SCALING LAW (Main Result):
   Operations ≈ {fit_params['coefficient']:.2e} × N^{fit_params['exponent']:.3f}
   
   Complexity: O(N^{fit_params['exponent']:.2f})
   Goodness of fit: R² = {fit_params['r_squared']:.6f} ({'Excellent' if fit_params['r_squared'] > 0.99 else 'Good' if fit_params['r_squared'] > 0.95 else 'Fair'})

2. DATA COLLECTION:
   • Tested {len(results)} composite numbers
   • Range: {results[0]['N']:,} to {results[-1]['N']:,}
   • Bit range: {results[0]['num_bits']} to {results[-1]['num_bits']} bits
   • Consistent 10 iterations per N
   • Average success rate: {avg_success_rate:.1f}%

3. EXTRAPOLATION TO RSA-2048:
   • RSA-2048 uses 2048-bit keys (≈617 decimal digits)
   • Predicted operations: ~10^{log_ops_2048:.0f}
   • Predicted time: ~10^{log_years_2048:.0f} years
   • This is ~10^{log_universe_ages_2048:.0f} times the age of the universe
   
4. CONCLUSION:
   Classical Shor's algorithm demonstrates polynomial scaling that
   makes RSA factorization computationally infeasible. The observed
   O(N^{fit_params['exponent']:.2f}) complexity ensures RSA security against classical attacks.
   
   However, quantum Shor's algorithm achieves O(log³N) complexity,
   making RSA vulnerable to quantum computers with sufficient qubits
   and error correction—highlighting the urgent need for post-quantum
   cryptography.

Copy the data arrays above for your plots!
    """)
    
    print("="*100 + "\n")

if __name__ == "__main__":
    main()


                    CLASSICAL SHOR'S ALGORITHM
               COMPREHENSIVE SCALING ANALYSIS WITH EXTRAPOLATION
                         10 ITERATIONS PER N

Configuration:
  Range: 15 to 10,000,000
  Samples: 200
  Iterations per N: 10 (consistent)
  Total iterations: 2,000

Generating 200 test numbers from 15 to 10,000,000...
Generated 187 unique test numbers
Actual range: 15 to 10,000,001
Bit range: 4 to 24 bits

Starting analysis...
Estimated runtime: 5-15 minutes (depends on your hardware)

[  1/187] Progress:   0.5% | Elapsed:    0.0s | ETA:    0.0s
[ 10/187] Progress:   5.3% | Elapsed:    0.0s | ETA:    0.0s
[ 20/187] Progress:  10.7% | Elapsed:    0.0s | ETA:    0.0s
[ 30/187] Progress:  16.0% | Elapsed:    0.0s | ETA:    0.0s
[ 40/187] Progress:  21.4% | Elapsed:    0.0s | ETA:    0.0s
[ 50/187] Progress:  26.7% | Elapsed:    0.0s | ETA:    0.0s
[ 60/187] Progress:  32.1% | Elapsed:    0.0s | ETA:    0.0s
[ 70/187] Progress:  37.4% | Elapsed:    0.1s | ETA:    0.1s
[ 80/187] 